In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ammarridho/drowsiness-cut")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'drowsiness-cut' dataset.
Path to dataset files: /kaggle/input/drowsiness-cut


In [ ]:
from tensorflow.keras.models import load_model
cnn_full = load_model("cnn_drowsiness_stage1.h5")

In [ ]:
cnn_full.build((None, 100, 100, 3))


In [ ]:
from tensorflow.keras.models import Model

cnn_feature_extractor = Model(
    inputs=cnn_full.layers[0].input,
    outputs=cnn_full.layers[-2].output
)


for layer in cnn_feature_extractor.layers:
    layer.trainable = False

cnn_feature_extractor.summary()



Model: "functional_33"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 100, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 98, 98, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 47, 47, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 21, 21, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 21, 21, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 8, 8, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 914,752 (3.49 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 914,752 (3.49 MB)

In [ ]:
def load_sequence_features_and_label(seq_path, cnn_classifier, cnn_feature_extractor):
    frames = sorted(os.listdir(seq_path))[:SEQ_LEN]
    images = []

    for frame in frames:
        img_path = os.path.join(seq_path, frame)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        images.append(img)

    images = np.array(images)

   
    frame_probs = cnn_classifier.predict(images, verbose=0)
    features = cnn_feature_extractor.predict(images, verbose=0)


    label = int(np.mean(frame_probs) > 0.5)

    return features, label


In [ ]:
def build_sequence_dataset(seq_root, cnn_classifier, cnn_feature_extractor):
    X, y = [], []

    seq_folders = sorted(os.listdir(seq_root))

    for i, seq_folder in enumerate(seq_folders):
        seq_path = os.path.join(seq_root, seq_folder)

        if os.path.isdir(seq_path):
            features, label = load_sequence_features_and_label(
                seq_path,
                cnn_classifier,
                cnn_feature_extractor
            )

            X.append(features)
            y.append(label)

        if i % 50 == 0:
            print(f"Processed {i}/{len(seq_folders)} sequences")

    return np.array(X), np.array(y)


In [ ]:
X_seq, y_seq = build_sequence_dataset(
   path+ "/test",
    cnn_classifier,
    cnn_feature_extractor
)

print(X_seq.shape)
print(y_seq.shape)


Processed 0/602 sequences
Processed 50/602 sequences
Processed 100/602 sequences
Processed 150/602 sequences
Processed 200/602 sequences
Processed 250/602 sequences
Processed 300/602 sequences
Processed 350/602 sequences
Processed 400/602 sequences
Processed 450/602 sequences
Processed 500/602 sequences
Processed 550/602 sequences
Processed 600/602 sequences
(602, 30, 128)
(602,)


In [ ]:

unique, counts = np.unique(y_seq, return_counts=True)
print(dict(zip(unique, counts)))


{np.int64(0): np.int64(167), np.int64(1): np.int64(435)}


In [ ]:
for i in [0, 1, 2]:
    print(f"Sequence {i}: label = {y_seq[i]}")


Sequence 0: label = 1
Sequence 1: label = 1
Sequence 2: label = 0


import numpy as np

np.save("X_seq_features.npy", X_seq)

np.save("y_seq_labels.npy", y_seq)

print("Sequence feature dataset saved")

from google.colab import files

files.download("X_seq_features.npy")

files.download("y_seq_labels.npy")

### if want to import those saved ones:

X_seq = np.load("X_seq_features.npy")

y_seq = np.load("y_seq_labels.npy")


np.save("cnn_pseudo_labels.npy", y_seq)

files.download("cnn_pseudo_labels.npy")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_seq,
    y_seq,
    test_size=0.2,
    random_state=42,
    stratify=y_seq   
)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)


(481, 30, 128) (481,)
(121, 30, 128) (121,)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))
print(class_weight_dict)


{np.int64(0): np.float64(1.8082706766917294), np.int64(1): np.float64(0.6910919540229885)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

lstm_model = Sequential([
    Input(shape=(30, 128)),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 139,905 (546.50 KB)

 Trainable params: 139,905 (546.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)


In [ ]:
history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=8,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.7782 - loss: 0.4384 - val_accuracy: 0.9256 - val_loss: 0.1476
Epoch 2/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9591 - loss: 0.1212 - val_accuracy: 0.9504 - val_loss: 0.1349
Epoch 3/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9499 - loss: 0.1074 - val_accuracy: 0.9752 - val_loss: 0.1234
Epoch 4/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9641 - loss: 0.0650 - val_accuracy: 0.9752 - val_loss: 0.0730
Epoch 5/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9434 - loss: 0.1185 - val_accuracy: 0.9339 - val_loss: 0.1632
Epoch 6/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9372 - loss: 0.1528 - val_accuracy: 0.9752 - val_loss: 0.0580
Epoch 7/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9569 - loss: 0.0747 - val_accuracy: 0.9752 - val_loss: 0.1137
Epoch 8/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9593 - loss: 0.1098 - val_accuracy: 0.9339 - val_loss

In [ ]:
y_val_prob = lstm_model.predict(X_val)
y_val_pred = (y_val_prob > 0.5).astype(int).ravel()


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_val, y_val_pred)
print(cm)


[[32  2]
 [ 1 86]]


In [ ]:
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Drowsy", "Non-Drowsy"]
))


              precision    recall  f1-score   support

      Drowsy       0.97      0.94      0.96        34
  Non-Drowsy       0.98      0.99      0.98        87

    accuracy                           0.98       121
   macro avg       0.97      0.96      0.97       121
weighted avg       0.98      0.98      0.98       121



In [ ]:
lstm_model.save("lstm_drowsiness_stage3.h5")
print("LSTM model saved")


LSTM model saved
